# Phase 0 — `WILLIAMS_COT_SWING_v1` validation

**Blocking gate** (PLAN §7, Phase 0): out-of-sample Sharpe ≥ 1.0 across ≥ 50% of the liquid universe.

If this notebook fails the gate, no UI is built. Jim Paul rule: define invalidation before you commit capital — same rule applies to engineering time.

Sources cited inline: Williams 1979, Briese 2008, Upperman 2008, Paul 1994.

**Data**: free CFTC disaggregated futures-only + free Stooq continuous-front-month bars. No vendor spend per PLAN §0.3 capital stop.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make `packages/` importable when the notebook is run from research/notebooks/.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'PLAN.md').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'packages'))

CACHE_DIR = REPO_ROOT / 'research' / 'data' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT, CACHE_DIR

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ingest.universe import UNIVERSE, sectors
from ingest import cftc_cot, prices, normalize, indicators, signal, backtest, metrics

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print(f'Universe: {len(UNIVERSE)} contracts across {len(sectors())} sectors')
for sec, contracts in sectors().items():
    print(f'  {sec:9s} {[c.symbol for c in contracts]}')

## 1. Ingest — CFTC + Stooq

Download every CFTC disaggregated futures-only ZIP from 2010 onward and every Stooq continuous-front-month price series for the universe. Everything is cached under `research/data/cache/` so re-runs are offline.

In [ ]:
YEARS = list(range(2010, 2026))
cot_raw = cftc_cot.load_universe(YEARS, UNIVERSE, CACHE_DIR)
print(f'COT rows: {len(cot_raw):,}')
cot_raw.head()

In [ ]:
px = prices.load_universe(UNIVERSE, CACHE_DIR)
print(f'Price rows: {len(px):,}')
print(f"Symbols with price data: {sorted(px['symbol'].unique())}")
px.groupby('symbol')['date'].agg(['min', 'max', 'count']).head(30)

## 2. Merge + indicators + signals

Per-symbol pipeline: forward-fill weekly COT onto daily bars, compute the six layer indicators, emit `entry_dir`.

In [ ]:
merged = normalize.join_cot_to_prices(px, cot_raw)

annotated_frames: dict[str, pd.DataFrame] = {}
for sym, grp in tqdm(list(merged.groupby('symbol')), desc='annotate'):
    if grp.empty or grp['net_commercials'].isna().all():
        continue
    g = indicators.add_all_indicators(grp.reset_index(drop=True))
    g = signal.annotate_signals(g)
    annotated_frames[sym] = g

print(f'Annotated {len(annotated_frames)} symbols')
sample_sym = next(iter(annotated_frames))
annotated_frames[sample_sym][['date', 'close', 'cot_index_comm', 'net_commercials', 'ucl', 'lcl', 'sma_fast', 'sma_slow', 'entry_dir']].tail(20)

### Signal-density sanity check

Six layers ANDed together should yield a *sparse* signal. Count entries per symbol — anything wildly off (zero across all markets, or thousands per market) signals an implementation bug, not an edge.

In [ ]:
density_rows = []
for sym, g in annotated_frames.items():
    density_rows.append({
        'symbol': sym,
        'bars': len(g),
        'long_signals': int(g['long_signal'].sum()),
        'short_signals': int(g['short_signal'].sum()),
        'any_entry': int((g['entry_dir'] != 0).sum()),
        'long_per_year': float(g['long_signal'].sum()) / max(1, len(g) / 252),
        'short_per_year': float(g['short_signal'].sum()) / max(1, len(g) / 252),
    })
density = pd.DataFrame(density_rows).sort_values('any_entry', ascending=False)
density

## 3. Walk-forward backtest

Per PLAN §1.6: 5-year warmup → 1-year out-of-sample, rolled annually. We score OOS only. The strategy has no hyperparameters to tune in v1 (literature values only), so the warmup window functions as indicator priming, not as a search.

In [ ]:
results: dict[str, backtest.BacktestResult] = {}
for sym, g in tqdm(list(annotated_frames.items()), desc='backtest'):
    contract = next(c for c in UNIVERSE if c.symbol == sym)
    results[sym] = backtest.walk_forward(g, contract)

total_trades = sum(len(r.trades) for r in results.values())
print(f'Backtested {len(results)} symbols; {total_trades} OOS trades total')

## 4. Scorecard

Per-contract scoring against PLAN §1.6 criteria. The right two columns are the pass flags: `scorecard_pass` is the strict §1.6 bar; `phase0_gate_pass` is the looser Phase 0 gate (Sharpe ≥ 1.0 AND ≥ 30 trades).

In [ ]:
cards = []
for sym, r in results.items():
    cards.append(metrics.score(sym, r.to_trades_df(), r.equity_curve))

scorecard_df = metrics.scorecard_to_dataframe(cards)
scorecard_df

## 5. Phase 0 gate

The single line below decides whether we proceed to Phase 1 — or stop, document, and re-route to LucidFlex per PLAN §0.3.

In [ ]:
gate = metrics.universe_gate(cards)
for k, v in gate.items():
    print(f'{k:22s} {v}')

if gate['passed']:
    print('\n[GATE] PASS — proceed to Phase 1 data spine.')
else:
    print('\n[GATE] FAIL — do NOT build UI. Document findings, re-route per stop-loss.')

## 6. Equity curves

Visual sanity. Look for: monotonic-ish growth in passing markets; flat/zigzag in failing ones; no single-trade dominance.

In [ ]:
passing = [s for s in cards if s.passes_gate]
fig, ax = plt.subplots(figsize=(11, 5))
for s in passing[:12]:
    eq = results[s.symbol].equity_curve
    if eq is None or eq.empty:
        continue
    normalized = eq / eq.iloc[0]
    ax.plot(eq.index, normalized, label=s.symbol, linewidth=1.1)
ax.set_title('OOS equity curves — gate-passing contracts (normalized to 1.0)')
ax.set_ylabel('Equity multiple')
ax.axhline(1.0, color='#888', linewidth=0.6)
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Per-market diagnostic — `entry_dir` overlay

Pick any symbol and visually verify entries align with COT extremes + trend confirmation. Useful both to catch implementation bugs and to feel the cadence the strategy trades at.

In [ ]:
DIAG_SYM = 'CL'  # change as needed
g = annotated_frames.get(DIAG_SYM)
if g is not None:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(g['date'], g['close'], color='#E8E8EA', linewidth=0.8, label='close')
    ax1.plot(g['date'], g['sma_slow'], color='#F5A623', linewidth=0.6, label='18 SMA')
    longs = g[g['entry_dir'] == 1]
    shorts = g[g['entry_dir'] == -1]
    ax1.scatter(longs['date'], longs['close'], marker='^', color='#18E08F', s=18, label='long entry')
    ax1.scatter(shorts['date'], shorts['close'], marker='v', color='#FF5A5F', s=18, label='short entry')
    ax1.set_title(f'{DIAG_SYM} — price + entry signals (Phase 0 diagnostic)')
    ax1.legend(fontsize=8)
    ax1.set_facecolor('#0A0A0B')
    ax2.plot(g['date'], g['cot_index_comm'], color='#18E08F', linewidth=0.8)
    ax2.axhline(80, color='#18E08F', linestyle='--', linewidth=0.5)
    ax2.axhline(20, color='#FF5A5F', linestyle='--', linewidth=0.5)
    ax2.set_ylabel('COT Index (commercials)')
    ax2.set_facecolor('#0A0A0B')
    for ax in (ax1, ax2):
        ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print(f'{DIAG_SYM} not in annotated set')

## 8. Trade log

Concatenated OOS trades. Sortable, exportable, the audit trail behind the scorecard. Inspect exit reasons — if `time_stop` dominates, the trend filter is letting in too many false starts.

In [ ]:
all_trades = pd.concat([r.to_trades_df() for r in results.values() if r.trades], ignore_index=True)
if not all_trades.empty:
    print(f'Total OOS trades: {len(all_trades)}')
    print('\nExit reason mix:')
    print(all_trades['reason'].value_counts())
    print('\nWin/loss split:')
    print((all_trades['pnl_usd'] > 0).value_counts())
all_trades.head(20)

## 9. Next actions

- If **GATE PASS**: write Phase 0 results into the master plan, commit + push, begin Phase 1 (R2 Parquet store + Friday cron).
- If **GATE FAIL**: do not build UI. Open a `findings.md`; document which markets failed and why (low signal density? bad trend regime? data gaps?). Then either iterate on v1.1 thresholds within the *literature* range, or stop and route focus back to LucidFlex per PLAN §0.3.
- Either way: ship a Twitter thread on the result (PLAN §7 Phase 6). Build in public.